In [5]:
import whisper
import os
import json
from dotenv import load_dotenv
load_dotenv()
import google.generativeai as genai
from groq import Groq
import torch
from tqdm.notebook import tqdm

In [74]:
os.system('python -m yt_dlp -x --audio-format mp3 -o "atv_20260429.mp3" https://www.youtube.com/watch?v=k7Cn_fCTX8g')

[youtube] Extracting URL: https://www.youtube.com/watch?v=k7Cn_fCTX8g
[youtube] k7Cn_fCTX8g: Downloading webpage


[youtube] k7Cn_fCTX8g: Downloading android vr player API JSON
[info] k7Cn_fCTX8g: Downloading 1 format(s): 251
[download] atv_20260429.mp3 has already been downloaded
[ExtractAudio] Not converting audio atv_20260429.mp3; file is already in target format mp3


0

### Original Whisper modell

In [75]:
model = whisper.load_model("medium") # a 'medium' már nagyon jó magyarul
result = model.transcribe("atv_20260429.mp3")

KeyboardInterrupt: 

In [ ]:
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

processor = AutoProcessor.from_pretrained("sarpba/whisper-base-hungarian_v1")
model = AutoModelForSpeechSeq2Seq.from_pretrained("sarpba/whisper-base-hungarian_v1")

Loading weights: 100%|██████████| 245/245 [00:00<00:00, 5393.03it/s]


### Hungarian Whisper modell

In [76]:
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

processor = AutoProcessor.from_pretrained("sarpba/whisper-base-hungarian_v1")
model = AutoModelForSpeechSeq2Seq.from_pretrained("sarpba/whisper-base-hungarian_v1")

Loading weights: 100%|██████████| 245/245 [00:00<00:00, 6034.79it/s]


In [ ]:
print(f"Használt szálak száma: {torch.get_num_threads()}")
# torch.set_num_threads(6)

Használt szálak száma: 6


In [81]:
import torch
from transformers import pipeline

# 1. Pipeline létrehozása a már betöltött modellel és processzorral
# A device="cpu" kényszeríti a processzoros futtatást
transcriber = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=30,  # Fontos: 30 másodperces ablakokban dolgozik
    device="cpu",
    batch_size = 6
)

# 2. Transzkripció - pont olyan egyszerű, mint korábban
# Az audio_path lehet a fájl neve (pl. "atv_20260429.mp3")
audio_file = "atv_20260429.mp3"

print("--- Transzkripció indítása (CPU-n)... ---")
result = transcriber(audio_file, generate_kwargs={"language": "hungarian"})

# 3. Az eredmény kiírása
print(result["text"])

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


--- Transzkripció indítása (CPU-n)... ---


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Az unió székhelyén tárgyal Magyar Péter a jogállami kifogások miatt, december 22-én hozza a nyilvánosságra, az ügynökaptákat a Tisza kormányban, a spontán privatizáció haszonélvezőit is feltárják. átlagosan 10-15%-kal drágulnak idén a strandbelépők, több helyen csak fünkös fordítnak a fürdők. a nyitott ablakon keresztül belőttek az egyik sojmáriobodába, közölte a Facebook oldalán a sojmári polgármesteri hivatal, amit a Telex vett észre, Azt írták, az ablak nyitva volt, de a redőnyt leengedték, a lövedék a redőnyt törte át, majd a helyiségőrös járőrei és a sojmári közterület felügyelő az eset után gyorsan kiért a helyszínre és előállította a feltételezett elkövetőt. októberben hozza nyilvánosságra az ügynökaktákat a Tisza kormány, Magyar Péter Facebook bejegyzésében azt ígéri, hogy az 1956-os forradalom hetedik évfordulója előtt egy nappal, október 22-én teljesítik a rendszerváltás egyik legfontosabb ígéretét, és nyilvánosságra hozzák valamint kereshetővé teszik a meglévő iratanyagot. a

In [85]:
result['text']

'Az unió székhelyén tárgyal Magyar Péter a jogállami kifogások miatt, december 22-én hozza a nyilvánosságra, az ügynökaptákat a Tisza kormányban, a spontán privatizáció haszonélvezőit is feltárják. átlagosan 10-15%-kal drágulnak idén a strandbelépők, több helyen csak fünkös fordítnak a fürdők. a nyitott ablakon keresztül belőttek az egyik sojmáriobodába, közölte a Facebook oldalán a sojmári polgármesteri hivatal, amit a Telex vett észre, Azt írták, az ablak nyitva volt, de a redőnyt leengedték, a lövedék a redőnyt törte át, majd a helyiségőrös járőrei és a sojmári közterület felügyelő az eset után gyorsan kiért a helyszínre és előállította a feltételezett elkövetőt. októberben hozza nyilvánosságra az ügynökaktákat a Tisza kormány, Magyar Péter Facebook bejegyzésében azt ígéri, hogy az 1956-os forradalom hetedik évfordulója előtt egy nappal, október 22-én teljesítik a rendszerváltás egyik legfontosabb ígéretét, és nyilvánosságra hozzák valamint kereshetővé teszik a meglévő iratanyagot. 

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 512, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(512, 512, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 512)
      (layers): ModuleList(
        (0-5): 6 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=False)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (fin

Save model's result to a JSON

In [7]:
with open('atv_20260429.json', 'w') as f:
    json.dump(result, f)

In [18]:
result=json.load(open('eredmeny_video_1.json'))['raw_text']

In [19]:
result

' Köszönöm önöket, ez az ATV híradó Benke Lucsia vagyok. Kárment és Brüsszselben ma az unió szék helyén tárgya a Magyar Péter a jogállami kifogások miatt lokkolt pénzekről. Október 22-én hozza nyilvánosságra, az ügynökaktákat a Tisza Kormány a spontán privatizáció haszonelvezőit is feltárják. Átlagosan 10-15%-kal drágulnak idén a strand belépők, több helyen csak pünkös fordítnak a fürdők. A nyitott ablakon keresztül belődtek az egyik sojmári obodába, közölte a Facebook oldalán a Sojmári Polgármesteri Hivatal, amit a Telex vett észre. Azt írták, az ablak nyitva volt, de a redönyk leengették. A lövedék a redönyk törte át, majd a helyiség faláról a földre esett. Személyi sérülés nem történt. A helyi rendőrőrs járőrei és a sojmári közterület felügyelő az eset után gyorsan kiért a helyszínre és előállította a feltételezett elkövetőd. Októberben hozza nyilvánosságra, az ügynökaktákat a Tisza Kormány. Magyar Péter Facebook bejegyzésében azt ígéri, hogy az 1956-os forradalom hetedik év forduló

In [ ]:
prompt = f"""
Feladat: Te egy professzionális szerkesztő vagy. Itt egy nyers híradó leirat, amit a 
Whisper készített. Javítsd ki a központozást és a helyesírást, és különösen figyelj a 
fonetikus félrehallások javítására (pl. "Atissa" -> "A Tisza", "sojmári" -> "solymári").

Kiemelten figyelj a nevekre, amik a magyar és nemzetközi politikában, közéletben gyakran
előfordulnak (pl. Vitézy Dávid, Lázár János, Magyar Péter, Tarr Zoltán, Gulyás Gergely, 
Szijjártó Péter). 

Ne hagyd ki a politikai kifejezéseket, és ügyelj arra, hogy a számok, dátumok logikusak 
legyenek, de maradj teljesen tárgyilagos. Ne írd át a mondatok eredeti jelentését, és 
semmiképpen ne találj ki (hallucinálj) új információkat vagy neveket a szövegbe! 
A könnyebb olvashatóság érdekében tagold a szöveget logikus bekezdésekre.

Csak a javított szöveget add vissza, mindenféle bevezető vagy magyarázat nélkül!

Szöveg: {result}
"""

In [26]:
api_key = os.getenv('GEMINI_API_KEY')
genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-flash-lite-latest')

In [25]:
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(f"Név: {m.name:35} | Verzió: {m.version}")

Név: models/gemini-2.5-flash             | Verzió: 001
Név: models/gemini-2.5-pro               | Verzió: 2.5
Név: models/gemini-2.0-flash             | Verzió: 2.0
Név: models/gemini-2.0-flash-001         | Verzió: 2.0
Név: models/gemini-2.0-flash-lite-001    | Verzió: 2.0
Név: models/gemini-2.0-flash-lite        | Verzió: 2.0
Név: models/gemini-2.5-flash-preview-tts | Verzió: gemini-2.5-flash-exp-tts-2025-05-19
Név: models/gemini-2.5-pro-preview-tts   | Verzió: gemini-2.5-pro-preview-tts-2025-05-19
Név: models/gemma-3-1b-it                | Verzió: 001
Név: models/gemma-3-4b-it                | Verzió: 001
Név: models/gemma-3-12b-it               | Verzió: 001
Név: models/gemma-3-27b-it               | Verzió: 001
Név: models/gemma-3n-e4b-it              | Verzió: 001
Név: models/gemma-3n-e2b-it              | Verzió: 001
Név: models/gemma-4-26b-a4b-it           | Verzió: 001
Név: models/gemma-4-31b-it               | Verzió: 001
Név: models/gemini-flash-latest          | Verzió: Gem

In [67]:
response = model.generate_content(prompt)

In [68]:
response.candidates

[content {
  parts {
    text: "Köszöntöm Önöket, ez az ATV Híradó, Benke Luca vagyok.\n\nKármentés Brüsszelben: ma az Unió székhelyén tárgyal Magyar Péter a jogállami kifogások miatt zárolt pénzekről. Október 22-én hozza nyilvánosságra az ügynökaktákat a Tisza-kormány, a „spontán privatizáció” haszonélvezőit is feltárják.\n\nÁtlagosan 10-15%-kal drágulnak idén a strandbelépők, több helyen csak pünkösdre nyitnak ki a fürdők.\n\n„A nyitott ablakon keresztül belőttek az egyik solymári óvodába” – közölte a Facebook-oldalán a Solymári Polgármesteri Hivatal, amit a Telex vett észre. Azt írták, az ablak nyitva volt, de a redőnyt leengedték. A lövedék a redőnyt törte át, majd a helyiség faláról a földre esett. Személyi sérülés nem történt. A helyi rendőrőrs járőrei és a solymári közterület-felügyelő az eset után gyorsan kiért a helyszínre, és előállította a feltételezett elkövetőt.\n\nOktóberben hozza nyilvánosságra az ügynökaktákat a Tisza-kormány. Magyar Péter Facebook-bejegyzésében azt ígé

In [69]:
print(response.candidates[0].content.parts[0].text)

Köszöntöm Önöket, ez az ATV Híradó, Benke Luca vagyok.

Kármentés Brüsszelben: ma az Unió székhelyén tárgyal Magyar Péter a jogállami kifogások miatt zárolt pénzekről. Október 22-én hozza nyilvánosságra az ügynökaktákat a Tisza-kormány, a „spontán privatizáció” haszonélvezőit is feltárják.

Átlagosan 10-15%-kal drágulnak idén a strandbelépők, több helyen csak pünkösdre nyitnak ki a fürdők.

„A nyitott ablakon keresztül belőttek az egyik solymári óvodába” – közölte a Facebook-oldalán a Solymári Polgármesteri Hivatal, amit a Telex vett észre. Azt írták, az ablak nyitva volt, de a redőnyt leengedték. A lövedék a redőnyt törte át, majd a helyiség faláról a földre esett. Személyi sérülés nem történt. A helyi rendőrőrs járőrei és a solymári közterület-felügyelő az eset után gyorsan kiért a helyszínre, és előállította a feltételezett elkövetőt.

Októberben hozza nyilvánosságra az ügynökaktákat a Tisza-kormány. Magyar Péter Facebook-bejegyzésében azt ígéri, hogy az 1956-os forradalom 69. évfor

In [57]:
load_dotenv()
client = Groq(api_key=os.getenv('GROQ_API_KEY'))

In [86]:
len(prompt[0:5000])

5000

In [113]:
completion = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
      {
        "role": "user",
        "content": prompt[0:10000]
      }
    ],
    temperature=1,
    max_completion_tokens=7000,
    top_p=1,
    stream=False,
    stop=None
)

In [114]:
groq_result2 = completion.choices[0].message.content

In [93]:
print(groq_result)

Köszönöm Önöket, ez az ATV híradó. Benke Lucia vagyok. 

Kárment és Brüsszelben ma az unió székhelyén tárgyal Magyar Péter a jogállami kifogások miatt blokkolt pénzekről. Október 22-én hozza nyilvánosságra a Tisza Kormány az ügynökaktákat, és a spontán privatizáció haszonélvezőit is feltárják. 

Átlagosan 10-15%-kal drágulnak idén a strandbelépők, több helyen csak pünkösd után nyitnak ki a fürdők. A nyitott ablakon keresztül lőttek bele az egyik szomori házba - közölte a Facebook-oldalán a Szomori Polgármesteri Hivatal, amit a Telex vett észre. Azt írták, az ablak nyitva volt, de a redőny leengedett. A lövedék a redőnyt törte át, majd a helyiség faláról a földre esett. Személyi sérülés nem történt. A helyi rendőrőrs járőrei és a szomori közterület-felügyelő az eset után gyorsan kiértek a helyszínre és előállították a feltételezett elkövetőt.

Októberben hozza nyilvánosságra az ügynökaktákat a Tisza Kormány. Magyar Péter Facebook-bejegyzésében azt ígéri, hogy az 1956-os forradalom heted

In [116]:
print(groq_result2)

Köszönöm önöket, ez az ATV híradó, Benke Lúcia vagyok. Kármen és Brüsszelben, ma az Európai Unió székhelyén tárgyalják a Magyar Péter által a jogállamiság kérdései miatt zárolt pénzeket. Október 22‑én hozza nyilvánosságra az ügy aktáit a Tisza‑Kormány, valamint a spontán privatizáció haszonélvezőit is feltárják.

Átlagosan 10‑15 %-kal drágulnak idén a strandbelépők; több helyen a fürdők a pünkösd előtt nyitnak.

A nyitott ablakon keresztül bejutottak a Sojmári polgármesteri irodába – közölte a Facebook oldalán a Sojmári Polgármesteri Hivatal, amit a Telex vett észre. Azt írták, az ablak nyitva volt, de a redőny leesett. A lövedék átszakította a redőnyt, majd a helyiség faláról a földre esett. Személyi sérülés nem történt. A helyi rendőrőrs járórei és a Sojmári közterület felügyelője az eset után gyorsan kiérkezett a helyszínre, és előállították a feltételezett elkövetőt.

Októberben hozza nyilvánosságra az ügy aktáit a Tisza‑Kormány. Magyar Péter Facebook‑bejegyzésében azt ígéri, hogy 

In [2]:
import bitsandbytes as bnb
print("BitsAndBytes verzió:", bnb.__version__)

BitsAndBytes verzió: 0.49.2


In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from dotenv import load_dotenv

load_dotenv(override=True)
my_token = os.getenv("HF_TOKEN")

model_name = "elte-nlp/Racka-4B"

# A token átadása közvetlenül a letöltőnek
tokenizer = AutoTokenizer.from_pretrained(
    model_name, 
    token=my_token
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=my_token, # Itt adjuk oda neki a kulcsot!
    dtype=torch.bfloat16,
    device_map="auto"
)

print("Siker! Megkezdődött a modell betöltése.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 398/398 [00:03<00:00, 118.86it/s]
Some parameters are on the meta device because they were offloaded to the disk.


Siker! Megkezdődött a modell betöltése.


In [24]:
from transformers import TextStreamer, GenerationConfig

messages = [
    {"role": "system", "content": "You are a helpful Hungarian assistant."},
    {"role": "user", "content": "Magyarázd el a gépi tanulás lényegét óvodásoknak egy mondatban!"}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)


generation_config = GenerationConfig(
    do_sample=True,
    temperature=0.6,
    top_p=0.8,
    top_k=50,
    repetition_penalty=1.1,
    presence_penalty=1.1,
)

streamer = TextStreamer(tokenizer, skip_prompt=True)

# conduct text completion
generated_ids = model.generate(
    input_ids = model_inputs["input_ids"],
    attention_mask = model_inputs["attention_mask"],
    max_new_tokens=200,
    generation_config=generation_config,
    streamer=streamer
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)

<think>
Okay, the user wants me to explain machine learning in one sentence for children. Let's see... Machine learning is when computers get better at doing things by looking at lots of examples and figuring out patterns on their own.


KeyboardInterrupt: 

In [23]:
from mistralai import Mistral

api_key = os.getenv("MISTRAL_API_KEY")
model = "mistral-large-2411"

client = Mistral(api_key=api_key)

chat_response = client.chat.complete(
    model=model,
    messages=[{"role": "user", "content": prompt}]
)

print(chat_response.choices[0].message.content)

Köszönöm, hogy velünk vannak, ez az ATV híradó, Benke Luca vagyok. Brüsszelben ma az Európai Unió székhelyén tárgyalják a Magyar Péter által jogállami kifogások miatt lefagyasztott pénzeket. Október 22-én hozza nyilvánosságra a Tisza Kormány az ügynökaktákat, és feltárják a spontán privatizáció haszonélvezőit. Átlagosan 10-15%-kal drágulnak idén a strandbelépők, több helyen csak pünkösdkor nyitnak a fürdők. A nyitott ablakon keresztül belődtek az egyik sojmári lakásba, közölte a Facebook oldalán a Sojmári Polgármesteri Hivatal, amit a Telex vett észre. Azt írták, az ablak nyitva volt, de a redőny le volt eresztve. A lövedék a redőnyt törte át, majd a helyiség faláról a földre esett. Személyi sérülés nem történt. A helyi rendőrség járőrei és a sojmári közterület-felügyelő az eset után gyorsan kiért a helyszínre és előállították a feltételezett elkövetőt.

Októberben hozza nyilvánosságra az ügynökaktákat a Tisza Kormány. Magyar Péter Facebook-bejegyzésében azt ígéri, hogy az 1956-os forr

In [ ]:
try:
    # Az összes elérhető modell lekérése
    models_response = client.models.list()

    print("--- Elérhető Mistral modellek ---")
    # A válasz egy objektum, aminek a 'data' attribútuma tartalmazza a listát
    for model in models_response.data:
        print(f"Modell ID: {model.id}")
        
except Exception as e:
    print(f"Hiba történt a lekérés során: {e}")

--- Elérhető Mistral modellek ---
Modell ID: mistral-medium-2505
Modell ID: mistral-medium-2508
Modell ID: mistral-medium-latest
Modell ID: mistral-medium
Modell ID: mistral-vibe-cli-with-tools
Modell ID: open-mistral-nemo
Modell ID: open-mistral-nemo-2407
Modell ID: mistral-tiny-2407
Modell ID: mistral-tiny-latest
Modell ID: codestral-2508
Modell ID: codestral-latest
Modell ID: devstral-2512
Modell ID: devstral-medium-latest
Modell ID: devstral-latest
Modell ID: mistral-small-2603
Modell ID: mistral-small-latest
Modell ID: mistral-vibe-cli-fast
Modell ID: magistral-small-latest
Modell ID: magistral-medium-2509
Modell ID: magistral-medium-latest
Modell ID: voxtral-small-2507
Modell ID: voxtral-small-latest
Modell ID: labs-leanstral-2603
Modell ID: mistral-large-2512
Modell ID: mistral-large-latest
Modell ID: mistral-large-2512
Modell ID: mistral-large-latest
Modell ID: ministral-3b-2512
Modell ID: ministral-3b-latest
Modell ID: ministral-8b-2512
Modell ID: ministral-8b-latest
Modell ID

### Azure STT

In [8]:
import azure.cognitiveservices.speech as speechsdk

In [16]:
# Cell for downloading the correct format
url = "https://www.youtube.com/watch?v=k7Cn_fCTX8g"
filename_wav = "atv_20260429.wav"

# We tell yt-dlp to output a standard WAV
# Azure loves 16000Hz mono, but standard 44.1/48kHz WAV usually works fine too
!python3 -m yt_dlp -x --audio-format wav -o {filename_wav} "{url}"

[youtube] Extracting URL: https://www.youtube.com/watch?v=k7Cn_fCTX8g
[youtube] k7Cn_fCTX8g: Downloading webpage
[youtube] k7Cn_fCTX8g: Downloading android vr player API JSON
[info] k7Cn_fCTX8g: Downloading 1 format(s): 251
[download] Destination: atv_20260429.webm
[download] 100% of   17.07MiB in 00:00:16 at 1.07MiB/s0;33m00:000m
[ExtractAudio] Destination: atv_20260429.wav
Deleting original file atv_20260429.webm (pass -k to keep)


In [17]:
speech_config = speechsdk.SpeechConfig(
            subscription=os.getenv('AZURE_SPEECH_KEY'), 
            region=os.getenv('AZURE_SPEECH_REGION')
            )
speech_config.speech_recognition_language = "hu-HU"
audio_config = speechsdk.audio.AudioConfig(filename=filename_wav)

In [18]:
recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, audio_config=audio_config)

In [19]:
import time

# Variable to hold our result
transcript_segments = []
done = False

def recognized_handler(evt):
    # This fires every time a full sentence is recognized
    if evt.result.text:
        print(f"Captured: {evt.result.text[:50]}...")
        transcript_segments.append(evt.result.text)

def stop_handler(evt):
    # This fires when the file ends or an error occurs
    global done
    print("Recognition finished or stopped.")
    done = True

# Connect the events to our recognizer object (from your previous cell)
recognizer.recognized.connect(recognized_handler)
recognizer.session_stopped.connect(stop_handler)
recognizer.canceled.connect(stop_handler)

In [21]:
recognizer.start_continuous_recognition()

while not done:
    time.sleep(1)

recognizer.stop_continuous_recognition()

Captured: Köszöntöm önöket ez az ATV híradó benke lucia vagy...
Captured: Kármentés Brüsszelbe ma az unió székhelyén tárgyal...
Captured: A nyitott ablakon keresztül belőttek az egyik soly...
Captured: Azt írták, az ablak nyitva volt, de a redőnyt leen...
Captured: Októberben hozza nyilvánosságra az ügynökaktákat a...
Captured: Egyeztetett cseh GERGŐ bendegúzzal az állambiztons...
Captured: Százszázalékos kármentésre nincs esély, de a befag...
Captured: Szerdán Brüsszelbe utazik és informális egyeztetés...
Captured: Vesztegetni való idő ezt magyar Péter jelentette b...
Captured: Sikeresek voltak arató László úgy véli, jelenleg M...
Captured: Hogy a helyreállítási források 10 egész négytized ...
Captured: Másrészt pedig más források is vannak befagyasztva...
Captured: Tehát, hogyha most a helyreállítási forrásokat sik...
Captured: Kiváltó dolgokat kell kezelni, tehát például....
Captured: Az állami egyetemek közérdekű vagyonkezelő alapítv...
Captured: 100 százalékos kártalanítás vagy...

KeyboardInterrupt: 

In [22]:
recognizer.stop_continuous_recognition()

In [ ]:
full_text = " ".join(transcript_segments)

In [25]:
full_text

'Köszöntöm önöket ez az ATV híradó benke lucia vagyok. Kármentés Brüsszelbe ma az unió székhelyén tárgyal magyar Péter a jogállami kifogások miatt blokkolt pénzekről. Okt. 22-én hozza nyilvánosságra az ügynökaktákat a Tisza kormány a spontán privatizáció haszonélvezőit is feltárják átlagosan tíz-tizenöt százalékkal drágulnak idén a strandbelépők több helyen csak pünkösd fordítják a fürdők. A nyitott ablakon keresztül belőttek az egyik solymári óvodába közölte a Facebook oldalán a solymári polgármesteri hivatal, amit a Telex vett észre. Azt írták, az ablak nyitva volt, de a redőnyt leengedték a lövedék a redőnyt törte át, majd a helyiség faláról a földre esett személyi sérülés nem történt. A helyi rendőrőrs járőrei és a solymári közterület felügyelő az eset után gyorsan kiért a helyszínre és előállította a feltételezett elkövetőt. Októberben hozza nyilvánosságra az ügynökaktákat a Tisza kormány magyar Péter Facebook bejegyzésében azt ígéri, hogy az ezerkilencszázötvenhatos forradalom 7.

In [27]:
prompt2 = f"""
Feladat: Te egy professzionális szerkesztő vagy. Itt egy nyers híradó leirat, amit a 
Whisper készített. Javítsd ki a központozást és a helyesírást, és különösen figyelj a 
fonetikus félrehallások javítására (pl. "Atissa" -> "A Tisza", "sojmári" -> "solymári").

Kiemelten figyelj a nevekre, amik a magyar és nemzetközi politikában, közéletben gyakran
előfordulnak (pl. Vitézy Dávid, Lázár János, Magyar Péter, Tarr Zoltán, Gulyás Gergely, 
Szijjártó Péter). 

Ne hagyd ki a politikai kifejezéseket, és ügyelj arra, hogy a számok, dátumok logikusak 
legyenek, de maradj teljesen tárgyilagos. Ne írd át a mondatok eredeti jelentését, és 
semmiképpen ne találj ki (hallucinálj) új információkat vagy neveket a szövegbe! 
A könnyebb olvashatóság érdekében tagold a szöveget logikus bekezdésekre.

Csak a javított szöveget add vissza, mindenféle bevezető vagy magyarázat nélkül!

Szöveg: {full_text}
"""

In [28]:
response = model.generate_content(prompt2)

In [31]:
print(response.candidates[0].content.parts[0].text)

Köszöntöm önöket, ez az ATV Híradó, Benke Lucia vagyok.

Kármentés Brüsszelben: ma az unió székhelyén tárgyal Magyar Péter a jogállami kifogások miatt blokkolt pénzekről. Október 22-én hozza nyilvánosságra az ügynökaktákat a Tisza-kormány; a spontán privatizáció haszonélvezőit is feltárják. Átlagosan tíz-tizenöt százalékkal drágulnak idén a strandbelépők, több helyen csak pünkösdkor nyitják meg a fürdőket.

A nyitott ablakon keresztül belőttek az egyik solymári óvodába – közölte a Facebook-oldalán a solymári polgármesteri hivatal, amit a Telex vett észre. Azt írták, az ablak nyitva volt, de a redőnyt leengedték; a lövedék a redőnyt törte át, majd a helyiség faláról a földre esett, személyi sérülés nem történt. A helyi rendőrőrs járőrei és a solymári közterület-felügyelő az eset után gyorsan kiért a helyszínre és előállította a feltételezett elkövetőt.

Októberben hozza nyilvánosságra az ügynökaktákat a Tisza-kormány. Magyar Péter Facebook-bejegyzésében azt ígéri, hogy az 1956-os forrad

### YouTube Transcription

In [3]:
pip install youtube-transcript-api

Note: you may need to restart the kernel to use updated packages.


In [15]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = "249z9TVsvho"

try:
    # API példány
    api = YouTubeTranscriptApi()

    # Elérhető feliratok listája
    transcript_list = api.list(video_id)

    # Automatikusan generált magyar felirat keresése
    transcript = transcript_list.find_generated_transcript(['hu'])

    # Felirat letöltése
    data = transcript.fetch()

    # Szöveg összefűzése
    full_text = " ".join(entry.text for entry in data)

    print("Sikerült! Íme a leirat eleje:")
    print("-" * 30)
    print(full_text[:500] + "...")

except Exception as e:
    print(f"Hiba történt: {e}")

Sikerült! Íme a leirat eleje:
------------------------------
Életbe lépett a tűzszünet az amerikai izraeli koalíció és Irán között. A történelmi energiaválság veszélye azonban még nem árult el. Megint követeli Brüsszel az üzemanyag védettárának kivezetését. Orbán Viktor azt mondta, ha engednének, az egekbe nőne a benzinára. Leleplező videót hozott nyilvánosságra. A NAV valóban korrupciós pénzt szállítottak az ukrán aranykonvolan. Már a visszatérésre készülnek a holdkerülés után az Artemis Misszió űrhajósai. Pénteken csapódhat az óceánba az Orion űrkapszul...


In [2]:
ydl_opts = {
    "quiet": True,
    "skip_download": True,
    "js_runtimes": {
        "node": "/Users/mac/.nvm/versions/node/v24.15.0/bin/node"
    }
}

In [9]:
ydl_opts = {
    "quiet": True,
    "skip_download": True,
    "js_runtimes": {
        "node": "/usr/local/bin/node"
    }
}

In [6]:
from youtube_transcript_api import YouTubeTranscriptApi
from yt_dlp import YoutubeDL

video_id = "249z9TVsvho"
video_url = f"https://www.youtube.com/watch?v={video_id}"

try:
    # =========================
    # Videó metaadatok
    # =========================
    ydl_opts = {
        "quiet": True,
        "skip_download": True,
    }

    with YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(video_url, download=False)

    print("Videó címe:", info.get("title"))
    print("Csatorna:", info.get("channel"))
    print("Feltöltő:", info.get("uploader"))
    print("Feltöltve:", info.get("upload_date"))
    print("Videó hossza:", info.get("duration"), "másodperc")

    print("-" * 50)

    # =========================
    # Transcript lekérés
    # =========================
    api = YouTubeTranscriptApi()

    transcript_list = api.list(video_id)

    transcript = transcript_list.find_generated_transcript(['hu'])

    data = transcript.fetch()

    full_text = " ".join(entry.text for entry in data)

    print("Leirat eleje:")
    print("-" * 30)
    print(full_text[:500] + "...")

except Exception as e:
    print(f"Hiba történt: {e}")

Videó címe: Híradó 2026.04.08. 19:30
Csatorna: M1 - Híradó
Feltöltő: M1 - Híradó
Feltöltve: 20260408
Videó hossza: 3285 másodperc
--------------------------------------------------
Leirat eleje:
------------------------------
Életbe lépett a tűzszünet az amerikai izraeli koalíció és Irán között. A történelmi energiaválság veszélye azonban még nem árult el. Megint követeli Brüsszel az üzemanyag védettárának kivezetését. Orbán Viktor azt mondta, ha engednének, az egekbe nőne a benzinára. Leleplező videót hozott nyilvánosságra. A NAV valóban korrupciós pénzt szállítottak az ukrán aranykonvolan. Már a visszatérésre készülnek a holdkerülés után az Artemis Misszió űrhajósai. Pénteken csapódhat az óceánba az Orion űrkapszul...


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.proxies import WebshareProxyConfig

ytt_api = YouTubeTranscriptApi(
    proxy_config=WebshareProxyConfig(
        proxy_username=os.environ('PROXY_USERNAME'),
        proxy_password=os.environ('PROXY_'),
    )
)

# all requests done by ytt_api will now be proxied through Webshare
ytt_api.fetch(video_id)

In [ ]:
import json
import os
import re
import sys
from tqdm import tqdm
from youtube_transcript_api import YouTubeTranscriptApi
from yt_dlp import YoutubeDL
import random
import time

# 1. DEBUGGING: Check Environment Variables
project_dir = os.getenv('PROJECT_DIR')
if project_dir is None:
    print("CRITICAL ERROR: 'PROJECT_DIR' is NOT set!")
    print("Try running: export PROJECT_DIR=$(pwd)  before running the script.")
    sys.exit(1)

print(f"DEBUG: Using PROJECT_DIR: {project_dir}")

# 2. Setup Paths
input_file = os.path.join(project_dir, 'video_links', 'video_ids.txt')
output_dir = os.path.join(project_dir, 'news_transcripts')

print(f"DEBUG: Looking for input file at: {input_file}")
if not os.path.exists(input_file):
    print(f"CRITICAL ERROR: Input file not found!")
    sys.exit(1)

if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"DEBUG: Created output directory: {output_dir}")

# 3. FIX JS RUNTIME: Search for Node.js path automatically
def get_node_path():
    # Common paths for node on macOS (Homebrew or Intel/Apple Silicon)
    paths = ['/usr/local/bin/node', '/opt/homebrew/bin/node', '/usr/bin/node']
    for path in paths:
        if os.path.exists(path):
            return path
    return None

node_path = get_node_path()

# 4. Extraction Logic
def sanitize_filename(name):
    return re.sub(r'[^\w\s-]', '', name).strip().replace(' ', '_')

with open(input_file, "r") as f:
    video_ids = [line.strip() for line in f if line.strip()]

# YT-DLP Configuration
ydl_opts = {
    "quiet": True,
    "skip_download": True,
    # This force-points yt-dlp to use Node.js if found
    "javascript_runtimes": [node_path] if node_path else None,
}

if node_path:
    print(f"DEBUG: Found JavaScript runtime at: {node_path}")
else:
    print("WARNING: Node.js not found! The extraction might fail.")

for v_id in tqdm(video_ids[50:53], desc="Processing Videos", unit="video"):
    time.sleep(random.uniform(1, 20))
    video_url = f"https://www.youtube.com/watch?v={v_id}"

    try:
        # Metadata Extraction
        with YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(video_url, download=False)
        
        channel_name = info.get("channel", "Unknown")
        upload_date = info.get("upload_date", "00000000")
        title = info.get("title", "")

        # Transcript Extraction
        try:
            api = YouTubeTranscriptApi()
            transcript_list = api.list(v_id)
            
            try:
                transcript = transcript_list.find_generated_transcript(['hu'])
            except:
                transcript = transcript_list.find_transcript(['hu'])
            
            data = transcript.fetch()
            full_text = " ".join(entry.text for entry in data)
            
        except Exception as te:
            tqdm.write(f"  - No transcript for {v_id}: {te}")
            full_text = None

        # Saving File
        if full_text:
            safe_channel = sanitize_filename(channel_name)
            filename = f"{safe_channel}_{upload_date}.json"
            filepath = os.path.join(output_dir, filename)

            output_data = {
                "video_id": v_id,
                "title": title,
                "channel": channel_name,
                "upload_date": upload_date,
                "transcript": full_text
            }

            with open(filepath, "w", encoding="utf-8") as json_file:
                json.dump(output_data, json_file, ensure_ascii=False, indent=4)
            
    except Exception as e:
        tqdm.write(f"  !!! Metadata Error for {v_id}: {str(e)[:80]}")

print("\n--- Done! ---")